# Trabalho 2 de Processamento de Linguagem Natural
## Coleta e pré-processamento de vagas de tecnologia

**Equipe:** Gabriel Schroeder Alfarth e Davi Deschamps

Execute as células em ordem. A coleta utiliza a API Adzuna e gera uma base bruta em JSONL e outra processada em CSV.

## 1. Instalação das bibliotecas

In [ ]:
!pip -q install requests pandas nltk

## 2. Credenciais da Adzuna
Crie gratuitamente o APP ID e a APP KEY em https://developer.adzuna.com/. Os valores digitados abaixo ficam ocultos.

In [ ]:
import os
from getpass import getpass

os.environ['ADZUNA_APP_ID'] = getpass('APP ID da Adzuna: ')
os.environ['ADZUNA_APP_KEY'] = getpass('APP KEY da Adzuna: ')
print('Credenciais registradas nesta sessão.')

APP ID da Adzuna: ··········
APP KEY da Adzuna: ··········
Credenciais registradas nesta sessão.


## 3. Implementação da coleta e do pré-processamento

In [ ]:
import hashlib
import json
import re
import time
import unicodedata
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests

API_URL = 'https://api.adzuna.com/v1/api/jobs/br/search/{pagina}'
CONSULTAS = [
    'desenvolvedor', 'analista de dados', 'cientista de dados',
    'devops', 'seguranca da informacao', 'suporte de tecnologia'
]
STOPWORDS_PT = {
    'a','ao','aos','aquela','aquelas','aquele','aqueles','aquilo','as','ate',
    'com','como','da','das','de','dela','dele','do','dos','e','ela','elas','ele',
    'eles','em','entre','era','essa','essas','esse','esses','esta','estas','este',
    'estes','eu','foi','for','ha','isso','isto','ja','mais','mas','me','mesmo',
    'meu','minha','muito','na','nas','nao','no','nos','nossa','nosso','o','os',
    'ou','para','pela','pelas','pelo','pelos','por','que','se','sem','ser','seu',
    'sua','tambem','tem','um','uma','voce'
}

def remover_ruidos(texto):
    texto = re.sub(r'<[^>]+>', ' ', texto or '')
    texto = re.sub(r'https?://\S+|www\.\S+', ' ', texto)
    texto = re.sub(r'\S+@\S+', ' ', texto)
    return re.sub(r'\s+', ' ', texto).strip()

def normalizar(texto):
    texto = remover_ruidos(texto).lower()
    texto = unicodedata.normalize('NFKD', texto)
    texto = ''.join(c for c in texto if not unicodedata.combining(c))
    texto = re.sub(r'[^a-z0-9+#.\s-]', ' ', texto)
    return re.sub(r'\s+', ' ', texto).strip()

def tokenizar(texto):
    return re.findall(r'(?:c\+\+|c#|\.net|[a-z0-9]+(?:[.-][a-z0-9]+)*)', texto)

def filtrar_tokens(tokens):
    return [t for t in tokens if t not in STOPWORDS_PT and (len(t) > 1 or t in {'c','r'})]

def aplicar_stemming(tokens):
    try:
        import nltk
        nltk.download('rslp', quiet=True)
        from nltk.stem import RSLPStemmer
        stemmer = RSLPStemmer()
        return [stemmer.stem(t) for t in tokens]
    except Exception:
        return tokens

def achatar_vaga(vaga, consulta):
    local = vaga.get('location') or {}
    empresa = vaga.get('company') or {}
    categoria = vaga.get('category') or {}
    texto_id = '|'.join(str(vaga.get(c, '')) for c in ('id','title','created','redirect_url'))
    return {
        'id_vaga': vaga.get('id') or hashlib.sha256(texto_id.encode()).hexdigest()[:20],
        'consulta_origem': consulta,
        'titulo': vaga.get('title', ''),
        'descricao': vaga.get('description', ''),
        'empresa': empresa.get('display_name', ''),
        'localizacao': local.get('display_name', ''),
        'categoria': categoria.get('label', ''),
        'salario_minimo': vaga.get('salary_min'),
        'salario_maximo': vaga.get('salary_max'),
        'tipo_contrato': vaga.get('contract_type', ''),
        'data_publicacao': vaga.get('created', ''),
        'url': vaga.get('redirect_url', ''),
        'coletado_em_utc': datetime.now(timezone.utc).isoformat()
    }

def coletar_vagas(paginas=2, resultados=50, intervalo=1.0):
    app_id = os.getenv('ADZUNA_APP_ID', '').strip()
    app_key = os.getenv('ADZUNA_APP_KEY', '').strip()
    if not app_id or not app_key:
        raise RuntimeError('Execute primeiro a célula das credenciais.')
    registros = []
    with requests.Session() as sessao:
        for consulta in CONSULTAS:
            print(f'Coletando: {consulta}')
            for pagina in range(1, paginas + 1):
                resposta = sessao.get(
                    API_URL.format(pagina=pagina),
                    params={'app_id': app_id, 'app_key': app_key, 'what': consulta,
                            'results_per_page': resultados, 'content-type': 'application/json',
                            'sort_by': 'date'},
                    timeout=30
                )
                resposta.raise_for_status()
                vagas = resposta.json().get('results', [])
                registros.extend(achatar_vaga(vaga, consulta) for vaga in vagas)
                time.sleep(intervalo)
    return registros

def preprocessar(registros):
    df = pd.DataFrame(registros)
    df = df.drop_duplicates(subset=['id_vaga'], keep='first')
    df = df[df['descricao'].fillna('').str.strip().ne('')].copy()
    df['texto_original'] = df['titulo'].fillna('').astype(str) + '. ' + df['descricao'].fillna('').astype(str)
    df['texto_sem_ruidos'] = df['texto_original'].map(remover_ruidos)
    df['texto_normalizado'] = df['texto_original'].map(normalizar)
    df['tokens'] = df['texto_normalizado'].map(tokenizar)
    df['tokens_sem_stopwords'] = df['tokens'].map(filtrar_tokens)
    df['texto_processado'] = df['tokens_sem_stopwords'].map(' '.join)
    df['stems'] = df['tokens_sem_stopwords'].map(aplicar_stemming).map(' '.join)
    return df

print('Funções carregadas com sucesso.')

Funções carregadas com sucesso.


## 4. Execução
A configuração abaixo busca até 600 registros brutos: seis consultas, duas páginas e 50 resultados por página.

In [ ]:
PAGINAS = 2
RESULTADOS_POR_PAGINA = 50

registros = coletar_vagas(PAGINAS, RESULTADOS_POR_PAGINA)
if not registros:
    raise RuntimeError('A API não retornou vagas para as consultas configuradas.')

df_processado = preprocessar(registros)
print(f'Registros brutos: {len(registros)}')
print(f'Vagas únicas processadas: {len(df_processado)}')
df_processado.head()

Coletando: desenvolvedor
Coletando: analista de dados
Coletando: cientista de dados
Coletando: devops
Coletando: seguranca da informacao
Coletando: suporte de tecnologia
Registros brutos: 600
Vagas únicas processadas: 568


,id_vaga,consulta_origem,titulo,descricao,empresa,localizacao,categoria,salario_minimo,salario_maximo,tipo_contrato,data_publicacao,url,coletado_em_utc,texto_original,texto_sem_ruidos,texto_normalizado,tokens,tokens_sem_stopwords,texto_processado,stems
0,5883958135,desenvolvedor,Desenvolvedor(a) de Software PHP / Python / SQ...,A Facta Promotora é uma instituição que se des...,Intera,"Porto Alegre, Rio Grande do Sul",Vagas em Tecnologia Informática,NaN,NaN,permanent,2026-09-14T19:42:44Z,https://www.adzuna.com.br/details/5883958135?u...,2026-09-15T00:09:35.887667+00:00,Desenvolvedor(a) de Software PHP / Python / SQ...,Desenvolvedor(a) de Software PHP / Python / SQ...,desenvolvedor a de software php python sql ser...,"[desenvolvedor, a, de, software, php, python, ...","[desenvolvedor, software, php, python, sql, se...",desenvolvedor software php python sql server f...,desenvolv softw php python sql serv fact promo...
1,5883943740,desenvolvedor,Desenvolvedor Full Stack Junior,Desenvolvedor Full Stack Junior OBRIGATÓRIOS: ...,GD9 Assessoria em Recursos Humanos,"Curitiba, Paraná",Unknown,NaN,NaN,,2026-09-14T19:29:51Z,https://www.adzuna.com.br/details/5883943740?u...,2026-09-15T00:09:35.887717+00:00,Desenvolvedor Full Stack Junior. Desenvolvedor...,Desenvolvedor Full Stack Junior. Desenvolvedor...,desenvolvedor full stack junior. desenvolvedor...,"[desenvolvedor, full, stack, junior, desenvolv...","[desenvolvedor, full, stack, junior, desenvolv...",desenvolvedor full stack junior desenvolvedor ...,desenvolv full stack juni desenvolv full stack...
2,5883943779,desenvolvedor,Desenvolvedor .NET Core - Fullstack,Requisitos necessários para esse novo desafio:...,Moot It Consulting,"São Paulo, Estado de São Paulo",Unknown,NaN,NaN,,2026-09-14T19:29:51Z,https://www.adzuna.com.br/details/5883943779?u...,2026-09-15T00:09:35.887732+00:00,Desenvolvedor .NET Core - Fullstack. Requisito...,Desenvolvedor .NET Core - Fullstack. Requisito...,desenvolvedor .net core - fullstack. requisito...,"[desenvolvedor, .net, core, fullstack, requisi...","[desenvolvedor, .net, core, fullstack, requisi...",desenvolvedor .net core fullstack requisitos n...,desenvolv .net cor fullstack requisit necessar...
3,5883943803,desenvolvedor,Desenvolvedor Web Jr,Nível: Analista Área: Tecnologia da Informação...,01 Tecnologia,"Curitiba, Paraná",Unknown,NaN,NaN,,2026-09-14T19:29:51Z,https://www.adzuna.com.br/details/5883943803?u...,2026-09-15T00:09:35.887744+00:00,Desenvolvedor Web Jr. Nível: Analista Área: Te...,Desenvolvedor Web Jr. Nível: Analista Área: Te...,desenvolvedor web jr. nivel analista area tecn...,"[desenvolvedor, web, jr, nivel, analista, area...","[desenvolvedor, web, jr, nivel, analista, area...",desenvolvedor web jr nivel analista area tecno...,desenvolv web jr nivel anal are tecnolog infor...
4,5883943732,desenvolvedor,Desenvolvedor Java Jr,"RESPONSABILIDADES E ATRIBUIÇÕES Desenvolver, e...",Projeto 22,"Rio de Janeiro, Estado do Rio de Janeiro",Unknown,NaN,NaN,,2026-09-14T19:29:51Z,https://www.adzuna.com.br/details/5883943732?u...,2026-09-15T00:09:35.887755+00:00,Desenvolvedor Java Jr. RESPONSABILIDADES E ATR...,Desenvolvedor Java Jr. RESPONSABILIDADES E ATR...,desenvolvedor java jr. responsabilidades e atr...,"[desenvolvedor, java, jr, responsabilidades, e...","[desenvolvedor, java, jr, responsabilidades, a...",desenvolvedor java jr responsabilidades atribu...,desenvolv jav jr respons atribuico desenvolv e...


## 5. Salvamento dos resultados

In [ ]:
pasta_raw = Path('/content/dados/raw')
pasta_processed = Path('/content/dados/processed')
pasta_raw.mkdir(parents=True, exist_ok=True)
pasta_processed.mkdir(parents=True, exist_ok=True)
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

arquivo_raw = pasta_raw / f'vagas_{timestamp}.jsonl'
with arquivo_raw.open('w', encoding='utf-8') as arquivo:
    for registro in registros:
        arquivo.write(json.dumps(registro, ensure_ascii=False) + '\n')

arquivo_csv = pasta_processed / f'vagas_processadas_{timestamp}.csv'
df_processado.to_csv(arquivo_csv, index=False, encoding='utf-8-sig')
print('Arquivos gerados:')
print(arquivo_raw)
print(arquivo_csv)

Arquivos gerados:
/content/dados/raw/vagas_20260915_001015.jsonl
/content/dados/processed/vagas_processadas_20260915_001015.csv


## 6. Download da entrega

In [ ]:
!zip -qr /content/dados_vagas.zip /content/dados
from google.colab import files
files.download('/content/dados_vagas.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>